# S-KeepTrack Training Pipeline
Ensure that your dataset is uploaded to Kaggle and update the `root_dir` in the training cell to point to your dataset's location.

In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.ops import roi_align
from tqdm.notebook import tqdm


In [ ]:
# --- dataloader.py ---
from torch.utils.data import Dataset

class TennisDataset(Dataset):
    def __init__(self, root_dir, mode='train', seq_len=3):
        self.root_dir = root_dir
        self.mode = mode
        self.seq_len = seq_len
        
        # 80/10/10 split
        if mode == 'train':
            self.games = [f'game{i}' for i in range(1, 9)]
        elif mode == 'val':
            self.games = ['game9']
        elif mode == 'test':
            self.games = ['game10']
        
        self.samples = self._parse_dataset()
        
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def _parse_dataset(self):
        samples = []
        for game in self.games:
            game_path = os.path.join(self.root_dir, game)
            if not os.path.exists(game_path):
                continue
            for clip in os.listdir(game_path):
                clip_path = os.path.join(game_path, clip)
                if not os.path.isdir(clip_path):
                    continue
                csv_path = os.path.join(clip_path, 'Label.csv')
                if not os.path.exists(csv_path):
                    continue
                
                df = pd.read_csv(csv_path)
                df = df[df['visibility'] > 0].reset_index(drop=True)
                
                for i in range(len(df) - self.seq_len + 1):
                    seq = []
                    for j in range(self.seq_len):
                        row = df.iloc[i+j]
                        img_path = os.path.join(clip_path, row['file name'])
                        x, y = int(row['x-coordinate']), int(row['y-coordinate'])
                        seq.append((img_path, x, y))
                    samples.append(seq)
        return samples

    def generate_heatmap(self, width, height, x, y, sigma=2):
        heatmap = np.zeros((height, width), dtype=np.float32)
        tmp_size = sigma * 3
        ul = [int(x - tmp_size), int(y - tmp_size)]
        br = [int(x + tmp_size + 1), int(y + tmp_size + 1)]
        
        size = 2 * tmp_size + 1
        x_mesh = np.arange(0, size, 1, np.float32)
        y_mesh = np.arange(0, size, 1, np.float32)
        x_mesh, y_mesh = np.meshgrid(x_mesh, y_mesh)
        
        gaussian = np.exp(- ((x_mesh - tmp_size)**2 + (y_mesh - tmp_size)**2) / (2 * sigma**2))
        
        g_x = max(0, -ul[0]), min(br[0], width) - ul[0]
        g_y = max(0, -ul[1]), min(br[1], height) - ul[1]
        
        img_x = max(0, ul[0]), min(br[0], width)
        img_y = max(0, ul[1]), min(br[1], height)
        
        if img_x[0] >= img_x[1] or img_y[0] >= img_y[1]:
            return heatmap
            
        heatmap[img_y[0]:img_y[1], img_x[0]:img_x[1]] = np.maximum(
            heatmap[img_y[0]:img_y[1], img_x[0]:img_x[1]],
            gaussian[g_y[0]:g_y[1], g_x[0]:g_x[1]]
        )
        return heatmap

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq = self.samples[idx]
        
        images = []
        heatmaps = []
        centers = []
        
        target_w, target_h = 640, 360
        
        for img_path, x, y in seq:
            img = cv2.imread(img_path)
            if img is None:
                img = np.zeros((1080, 1920, 3), dtype=np.uint8)
            else:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                
            orig_h, orig_w = img.shape[:2]
            scale_x = target_w / orig_w
            scale_y = target_h / orig_h
            
            img = cv2.resize(img, (target_w, target_h))
            
            scaled_x = int(x * scale_x)
            scaled_y = int(y * scale_y)
            
            heatmap = self.generate_heatmap(target_w // 4, target_h // 4, scaled_x // 4, scaled_y // 4)
            
            images.append(self.transform(img))
            heatmaps.append(torch.tensor(heatmap).unsqueeze(0))
            centers.append(torch.tensor([scaled_x, scaled_y], dtype=torch.float32))
            
        return {
            'images': torch.stack(images),
            'heatmaps': torch.stack(heatmaps),
            'centers': torch.stack(centers)
        }


In [ ]:
# --- backbone.py ---

class Backbone(nn.Module):
    def __init__(self):
        super(Backbone, self).__init__()
        # Use a pretrained ResNet50 as base
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        
        # Low-level Branch: preserve spatial features (e.g. up to layer2, feature stride 8)
        self.low_level = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2
        )
        
        # High-level Branch: layer3 (feature stride 16)
        self.high_level = resnet.layer3

    def forward(self, x):
        feat_low = self.low_level(x)
        feat_high = self.high_level(feat_low)
        return feat_low, feat_high


In [ ]:
# --- candidate_extraction.py ---

class TargetClassifier(nn.Module):
    def __init__(self, in_channels=1024):
        super(TargetClassifier, self).__init__()
        # Several 3x3 Convs to process high-level features into a heatmap
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.conv(x)

def extract_candidates(heatmap, top_k=10, pool_size=3):
    """
    Extract top_k local maxima from the heatmap using MaxPool2d as NMS.
    """
    B, C, H, W = heatmap.shape
    pad = pool_size // 2
    # NMS via MaxPool
    hmax = F.max_pool2d(heatmap, pool_size, stride=1, padding=pad)
    keep = (hmax == heatmap).float()
    heatmap = heatmap * keep
    
    heatmap = heatmap.view(B, -1) # [B, H*W]
    top_scores, top_indices = torch.topk(heatmap, top_k, dim=1)
    
    top_y = torch.div(top_indices, W, rounding_mode='floor')
    top_x = top_indices % W
    
    candidates = torch.stack([top_x, top_y], dim=-1).float() # [B, top_k, 2]
    return candidates, top_scores


In [ ]:
# --- feature_encoding.py ---

class FeatureEncoding(nn.Module):
    def __init__(self, feat_stride, roi_size=5):
        super(FeatureEncoding, self).__init__()
        self.feat_stride = feat_stride
        self.roi_size = roi_size

    def forward(self, feature_map, candidates):
        """
        feature_map: [B, C, H, W]
        candidates: [B, N, 2] (x, y) coordinates relative to input image scale. 
        """
        B, N, _ = candidates.shape
        C = feature_map.shape[1]
        
        # Prepare rois for roi_align: [batch_index, x1, y1, x2, y2]
        box_size = 4.0 # pixels to crop in feature space
        
        rois = []
        for b in range(B):
            for n in range(N):
                cx, cy = candidates[b, n, 0], candidates[b, n, 1]
                x1, y1 = cx - box_size/2, cy - box_size/2
                x2, y2 = cx + box_size/2, cy + box_size/2
                rois.append([b, x1, y1, x2, y2])
                
        rois = torch.tensor(rois, dtype=torch.float32, device=feature_map.device)
        
        # Apply RoI Align
        aligned_features = roi_align(
            feature_map, 
            rois, 
            output_size=(self.roi_size, self.roi_size), 
            spatial_scale=1.0  # Assumes candidates passed match the scale of the feature map
        )
        
        aligned_features = aligned_features.view(B, N, C, self.roi_size, self.roi_size)
        
        # flatten the spatial dimensions of RoI
        return aligned_features.view(B, N, -1)


In [ ]:
# --- candidate_embedding.py ---

class CandidateEmbedding(nn.Module):
    def __init__(self, in_dim, embed_dim=256, num_heads=4, num_layers=2):
        super(CandidateEmbedding, self).__init__()
        # Project RoI features down to embed length
        self.proj = nn.Linear(in_dim, embed_dim)
        
        # Self-attention module to learn spatio-temporal relationships between candidates
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dim_feedforward=embed_dim*2, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        """
        x: [B, N, in_dim]
        """
        x = self.proj(x)
        x = self.transformer(x)
        return x


In [ ]:
# --- association.py ---

class ObjectAssociation(nn.Module):
    def __init__(self, embed_dim=256):
        super(ObjectAssociation, self).__init__()
        self.w_low = nn.Parameter(torch.tensor(0.0)) # Starts at logit 0 -> sigmoid 0.5

    def compute_association_matrix(self, embed_t_minus_1, embed_t):
        """
        embed: [B, N, embed_dim]
        Returns: [B, N, N] association matrix
        """
        # Candidate correlation between frames
        matrix = torch.bmm(embed_t_minus_1, embed_t.transpose(1, 2))
        return matrix

    def forward(self, low_t1, low_t, high_t1, high_t):
        # Calculate parallel associations from both branches
        A_low = self.compute_association_matrix(low_t1, low_t)
        A_high = self.compute_association_matrix(high_t1, high_t)
        
        # Weighted Fusion 
        w = torch.sigmoid(self.w_low)
        A = w * A_low + (1 - w) * A_high
        A = F.softmax(A, dim=-1) # Softmax over candidates in current frame t
        
        return A


In [ ]:
# --- loss.py ---

def focal_loss(pred, target, alpha=2, beta=4):
    """
    pred: [B, C, H, W]
    target: [B, C, H, W] Gaussian heatmap
    """
    pos_inds = target.eq(1).float()
    neg_inds = target.lt(1).float()
    
    neg_weights = torch.pow(1 - target, beta)
    
    pred = torch.clamp(pred, 1e-4, 1 - 1e-4)
    
    pos_loss = torch.log(pred) * torch.pow(1 - pred, alpha) * pos_inds
    neg_loss = torch.log(1 - pred) * torch.pow(pred, alpha) * neg_weights * neg_inds
    
    num_pos = pos_inds.float().sum()
    pos_loss = pos_loss.sum()
    neg_loss = neg_loss.sum()
    
    if num_pos == 0:
        return -neg_loss
    return -(pos_loss + neg_loss) / num_pos


def association_loss(A, current_gt_index, prev_gt_index):
    """
    Cross-entropy equivalent penalizing incorrect matches
    A: [B, N, N] - Association scores between frame t-1 and frame t
    current_gt_index: [B] - Index of true candidate in frame t
    prev_gt_index: [B] - Index of true candidate in frame t-1
    """
    B = A.shape[0]
    loss = 0.0
    for b in range(B):
        true_t1 = prev_gt_index[b]
        true_t = current_gt_index[b]
        
        if true_t1 < A.shape[1] and true_t < A.shape[2]:
            score = A[b, true_t1, true_t]
            loss += -torch.log(score + 1e-6)
    return loss / B


In [ ]:
# --- train.py ---
from torch.utils.data import DataLoader


def find_gt_candidate_index(candidates, gt_center, stride=16):
    gt = gt_center.unsqueeze(1) / stride
    dists = torch.norm(candidates - gt, dim=-1)
    return dists.argmin(dim=-1)

def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    root_dir = '/kaggle/input/tennis-ball-dataset/Dataset'  # UPDATE THIS FOR KAGGLE
    
    try:
        train_dataset = TennisDataset(root_dir, mode='train', seq_len=3)
        train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0)
        
        val_dataset = TennisDataset(root_dir, mode='val', seq_len=3)
        val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=0)
    except Exception as e:
        print(f"Error initializing dataloader: {e}")
        train_loader = []
        val_loader = []

    # ... (modules and optimizer initialization)
    # [Rest of module initialization remains the same]
    backbone = Backbone().to(device)
    target_classifier = TargetClassifier(in_channels=1024).to(device)
    feat_encode_low = FeatureEncoding(feat_stride=8, roi_size=5).to(device)
    feat_encode_high = FeatureEncoding(feat_stride=16, roi_size=5).to(device)
    embed_low = CandidateEmbedding(in_dim=512*5*5, embed_dim=256).to(device)
    embed_high = CandidateEmbedding(in_dim=1024*5*5, embed_dim=256).to(device)
    association = ObjectAssociation(embed_dim=256).to(device)
    
    optimizer = optim.Adam(
        list(backbone.parameters()) + 
        list(target_classifier.parameters()) +
        list(feat_encode_low.parameters()) +
        list(feat_encode_high.parameters()) +
        list(embed_low.parameters()) +
        list(embed_high.parameters()) +
        list(association.parameters()), 
        lr=1e-4
    )

    epochs = 10
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # --- TRAINING PHASE ---
        print(f"Epoch [{epoch+1}/{epochs}] - Training")
        backbone.train()
        target_classifier.train()
        
        total_train_loss = 0
        pbar = tqdm(train_loader) if len(train_loader) > 0 else []
        for batch_idx, batch in enumerate(pbar):
            images = batch['images'].to(device)
            heatmaps = batch['heatmaps'].to(device)
            centers = batch['centers'].to(device)
            B, T, C, H, W = images.shape
            
            optimizer.zero_grad()
            batch_loss = 0
            prev_low, prev_high = None, None
            prev_gt_idx = None
            
            for t in range(T):
                img_t = images[:, t] 
                gt_heatmap_t = heatmaps[:, t]
                feat_low, feat_high = backbone(img_t)
                pred_heatmap = target_classifier(feat_high)
                gt_heatmap_resized = torch.nn.functional.interpolate(gt_heatmap_t, size=pred_heatmap.shape[-2:], mode='bilinear', align_corners=False)
                loss_hm = focal_loss(pred_heatmap, gt_heatmap_resized)
                
                candidates, _ = extract_candidates(pred_heatmap, top_k=10)
                curr_gt_idx = find_gt_candidate_index(candidates, centers[:, t], stride=16)
                
                roi_feat_low = feat_encode_low(feat_low, candidates * 2.0)
                roi_feat_high = feat_encode_high(feat_high, candidates)
                emb_low, emb_high = embed_low(roi_feat_low), embed_high(roi_feat_high)
                
                if t > 0 and prev_low is not None:
                    A = association(prev_low, emb_low, prev_high, emb_high)
                    loss_assoc = association_loss(A, current_gt_index=curr_gt_idx, prev_gt_index=prev_gt_idx)
                    batch_loss += loss_hm + loss_assoc
                else:
                    batch_loss += loss_hm
                prev_low, prev_high = emb_low, emb_high
                prev_gt_idx = curr_gt_idx
                
            batch_loss.backward()
            optimizer.step()
            total_train_loss += batch_loss.item()
            if isinstance(pbar, tqdm): pbar.set_description(f"Train Loss: {batch_loss.item():.4f}")

        # --- VALIDATION PHASE ---
        print(f"Epoch [{epoch+1}/{epochs}] - Validation")
        backbone.eval()
        target_classifier.eval()
        total_val_loss = 0
        
        with torch.no_grad():
            vbar = tqdm(val_loader) if len(val_loader) > 0 else []
            for batch in vbar:
                images, heatmaps, centers = batch['images'].to(device), batch['heatmaps'].to(device), batch['centers'].to(device)
                B, T, C, H, W = images.shape
                v_loss = 0
                prev_low, prev_high = None, None
                prev_gt_idx = None
                for t in range(T):
                    feat_low, feat_high = backbone(images[:, t])
                    pred_heatmap = target_classifier(feat_high)
                    gt_hm = torch.nn.functional.interpolate(heatmaps[:, t], size=pred_heatmap.shape[-2:], mode='bilinear', align_corners=False)
                    loss_hm = focal_loss(pred_heatmap, gt_hm)
                    candidates, _ = extract_candidates(pred_heatmap, top_k=10)
                    curr_gt_idx = find_gt_candidate_index(candidates, centers[:, t], stride=16)
                    roi_low = feat_encode_low(feat_low, candidates * 2.0)
                    roi_high = feat_encode_high(feat_high, candidates)
                    emb_low, emb_high = embed_low(roi_low), embed_high(roi_high)
                    if t > 0 and prev_low is not None:
                        A = association(prev_low, emb_low, prev_high, emb_high)
                        v_loss += loss_hm + association_loss(A, current_gt_index=curr_gt_idx, prev_gt_index=prev_gt_idx)
                    else:
                        v_loss += loss_hm
                    prev_low, prev_high = emb_low, emb_high
                    prev_gt_idx = curr_gt_idx
                total_val_loss += v_loss.item()
                if isinstance(vbar, tqdm): vbar.set_description(f"Val Loss: {v_loss.item():.4f}")

        avg_train = total_train_loss / (len(train_loader) + 1e-5)
        avg_val = total_val_loss / (len(val_loader) + 1e-5)
        print(f"Avg Train Loss: {avg_train:.4f} | Avg Val Loss: {avg_val:.4f}")
        
        checkpoint = {
            'epoch': epoch,
            'backbone': backbone.state_dict(),
            'target_classifier': target_classifier.state_dict(),
            'feat_encode_low': feat_encode_low.state_dict(),
            'feat_encode_high': feat_encode_high.state_dict(),
            'embed_low': embed_low.state_dict(),
            'embed_high': embed_high.state_dict(),
            'association': association.state_dict(),
            'val_loss': avg_val
        }
        
        # Save Latest
        torch.save(checkpoint, "latest_model.pth")
        
        # Save Best
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(checkpoint, "best_model.pth")
            print(f"New Best Model saved (Val Loss: {avg_val:.4f})")


if __name__ == '__main__':
    train()


In [ ]:
# Run the training loop
# Ensure Kaggle Hardware Accelerator is set to 'GPU T4x2' or 'P100'
if __name__ == '__main__':
    train()
